### files to staging(bronze) to target_table(silver)

In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/inbound/emp_details.csv")

In [0]:
df2=spark.read.format("csv").option("header","true").option("inferSchema","true").load("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/inbound/dept_details.csv")

In [0]:
df2.write.format("delta").mode("overwrite").saveAsTable("shivasai_catalog.bronze.department_details")

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("shivasai_catalog.bronze.employee_details")

In [0]:
%sql
create or replace table shivasai_catalog.silver.employee_details
using delta 
as
select 
e.empid,
concat(e.firstname,' ',e.lastname) as fullname,
e.age,
e.salary,
e.salary*0.10 as bonus,
e.salary+bonus as total_salary,
case 
when  e.salary > 5000 then "high_salaried_employee" 
when e.salary > 3000 and  e.salary <= 5000 then "medium_salaried_employee"
when e.salary <= 3000 then "low_salaried_employee"
else "no salary"
end as salary_catagory,

e.gender,
case
when e.gender='M' then 'Male'
when e.gender='F' then 'Female'
else 'NA'
end as gender_desc,
e.deptno,
d.deptname,
e.state,
e.mobilenumber

from shivasai_catalog.bronze.employee_details e
left join shivasai_catalog.bronze.department_details d
on e.deptno = d.deptid

In [0]:
%sql
select * from shivasai_catalog.silver.employee_details